In [1]:
from langgraph.graph import StateGraph ,START,END 

from langchain_groq import ChatGroq
from dotenv import load_dotenv 
load_dotenv() 
from typing import TypedDict,Annotated 
from langchain_core.messages import HumanMessage,BaseMessage

from langgraph.checkpoint.memory import MemorySaver  




In [2]:
llm=ChatGroq(model='Llama-3.3-70b-Versatile') 

In [3]:
# llm.invoke('hi') for checking the llm api 


In [4]:
from langgraph.graph.message import add_messages
#here BaseMessage is abstract it can be any kind of message human,system,ai,etc 
# add_messages is a reducer by which we append the messages 

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [5]:
#lets make a obejct for the persistence memory 
checkpointer=MemorySaver() 

#lets make our graph 
graph=StateGraph(ChatState) 


In [6]:
#logic for the chatting 
def Chat_with_LLM(state: ChatState)-> ChatState: 
    response=llm.invoke(state['messages']) 
    return {'messages':[response]}

In [7]:
#lets add node thier is only 1 node in it 
#by which user can communicate with the llm 
graph.add_node('Chat_Node',Chat_with_LLM) 




#lets add edges 
graph.add_edge(START,'Chat_Node') 
graph.add_edge('Chat_Node',END)




In [8]:
chat_bot=graph.compile(checkpointer=checkpointer) 


In [9]:
msg={'messages':HumanMessage(content='Hey make the outline for a blog about llm.')}
# final_output=chat_bot.invoke(msg) 



In [10]:
# ai_response=final_output['messages'][-1].content

In [11]:
from IPython.display import display,Markdown 
# display(Markdown(ai_response))


In [12]:
#lets provide the ability like a chatbot 
while True: 
    user_msg=input('Enter your message: ') 
    print(f'User: {user_msg}') 
    
    if user_msg.strip().lower() in ['exit','quit','stop']: 
        print('Exiting the chat') 
        
        break 
    
    # response=chat_bot.invoke({'messages':[HumanMessage(content=user_msg)]})
    # ai_response=response['messages'][-1].content 
    
    # print(f'AI: {ai_response}') 
    #lets print the proper message using the Ipython 
    # display(Markdown(f'**AI:** {ai_response}'))
    
    #here in the above logic our chatbot have no memory so lets add history 
    #because we are invoking this in every invocation 
    

User: 
User: exit
Exiting the chat


In [13]:
#lets see Persistence of the chatbot 
#here in the above logic we are always giving new state as input 
#but now we store that previous state and use it in the next invocation
# from langgraph.checkpoint.memory import MemorySaver 

#at the time of invocation we have to define the thread 
thread_id='1'
while True: 
    user_msg=input('Enter your message: ') 
    print(f'User: {user_msg}') 
    
    if user_msg.strip().lower() in ['exit','quit','stop']: 
        print('Exiting the chat') 
        
        break 
    config={'configurable':{'thread_id':thread_id}}
    
    response=chat_bot.invoke({'messages':[HumanMessage(content=user_msg)]},config=config)
    ai_response=response['messages'][-1].content 
    
    # print(f'AI: {ai_response}') 
    #lets print the proper message using the Ipython 
    display(Markdown(f'**AI:** {ai_response}'))
    
    #here in the above logic our chatbot have no memory so lets add history 
    #because we are invoking this in every invocation 
    

User: exit
Exiting the chat


In [19]:
#lets see hoW to ADD The sTREAMING InsteAD Of providing all the response at a time 
#instread of invoke we have to use the stream 
#and provide the initial state 
config={'configurable':{'thread_id':'1111'}}
#the function will return a generator 
#we have to use the for loop to get the tokens from the generator one by one 
stream_generator= chat_bot.stream({'messages':[HumanMessage(content='make a blog about llm')]},config=config,stream_mode='messages')
    
# for token in stream_generator:

for msg_chunk,meta_data in stream_generator:
    
    if msg_chunk.content: 
        print(msg_chunk.content,end='',flush=True)

**The Power of Large Language Models (LLMs): Unlocking the Future of Artificial Intelligence**

Introduction

In recent years, the field of artificial intelligence (AI) has witnessed a significant breakthrough with the emergence of Large Language Models (LLMs). These models have revolutionized the way we interact with machines, enabling them to understand and generate human-like language. In this blog, we will delve into the world of LLMs, exploring their capabilities, applications, and the potential impact they may have on our daily lives.

**What are Large Language Models (LLMs)?**

Large Language Models are a type of artificial intelligence designed to process and generate human language. They are trained on vast amounts of text data, which enables them to learn patterns, relationships, and context. This training allows LLMs to generate coherent and natural-sounding text, making them incredibly useful for a wide range of applications.

**How do LLMs work?**

LLMs work by using a com

In [ ]:
print(type(stream_generator))


<class 'generator'>
